In [5]:
# ==============================
# Image Registration and Field‑of‑View Alignment
# ==============================

import os
import re
import math
import json
import csv
from pathlib import Path
from typing import Dict, List, Tuple
import numpy as np
from PIL import Image

# ==============================
# USER-DEFINED FOLDERS
# ==============================
ROOT = Path(r"C:\Your_Input_directory") # Specify input directory
OUT_ROOT = Path(r"C:\Your_Output_directory") #  Specify output directory

PATTERN = "s1_w1"
REF_C = "c1"

# =========================================
# Helper functions
# =========================================

C_FOLDER_REGEX = re.compile(r"^c\d+$", re.IGNORECASE)
TIFF_EXTS = {".tif", ".tiff"}

def is_tiff(p: Path) -> bool:
    return p.is_file() and p.suffix.lower() in TIFF_EXTS

def load_grayscale_float(image_path: Path) -> np.ndarray:
    with Image.open(image_path) as im:
        im.seek(0)
        im_f = im.convert("F")
        arr = np.array(im_f, dtype=np.float32)
    return arr

def phase_correlation_shift(ref: np.ndarray, img: np.ndarray) -> Tuple[int, int]:
    H, W = ref.shape

    def _prep(a):
        a = a - np.mean(a)
        wy = np.hanning(H).astype(np.float32)
        wx = np.hanning(W).astype(np.float32)
        return a * np.outer(wy, wx)

    A = _prep(ref)
    B = _prep(img)

    FA = np.fft.fft2(A)
    FB = np.fft.fft2(B)
    R = FA.conj() * FB
    R /= np.maximum(np.abs(R), 1e-12)

    cc = np.abs(np.fft.ifft2(R))
    py, px = np.unravel_index(np.argmax(cc), cc.shape)

    if py > H // 2: py -= H
    if px > W // 2: px -= W

    return int(py), int(px)

def find_c_groups(root: Path) -> List[Path]:
    groups = []
    for dirpath, dirnames, _ in os.walk(root):
        c_dirs = [d for d in dirnames if C_FOLDER_REGEX.match(d)]
        if len(c_dirs) >= 2:
            groups.append(Path(dirpath))
    return groups

def list_c_folders(group_dir: Path) -> List[Path]:
    out = []
    for d in group_dir.iterdir():
        if d.is_dir() and C_FOLDER_REGEX.match(d.name):
            out.append(d)

    def _key(p: Path):
        m = re.search(r"\d+", p.name)
        return int(m.group()) if m else 0

    return sorted(out, key=_key)

def collect_pattern_filenames(cdir: Path, pattern: str):
    files = []
    for p in cdir.iterdir():
        if is_tiff(p) and pattern.lower() in p.name.lower():
            files.append(p.name)
    return sorted(files)

def compute_folder_shifts(c_folders, ref_name, pattern):
    ref_c = [p for p in c_folders if p.name.lower() == ref_name.lower()][0]

    pattern_files = collect_pattern_filenames(ref_c, pattern)
    if not pattern_files:
        raise RuntimeError(f"No files with pattern {pattern} found in {ref_c}")

    ref_img0 = load_grayscale_float(ref_c / pattern_files[0])
    H, W = ref_img0.shape

    shifts = {ref_c.name: (0, 0)}

    for cdir in c_folders:
        if cdir == ref_c:
            continue

        collected = []
        for fname in pattern_files:
            if not (cdir / fname).exists():
                continue
            ref_img = load_grayscale_float(ref_c / fname)
            img = load_grayscale_float(cdir / fname)

            dy, dx = phase_correlation_shift(ref_img, img)

            collected.append((-dy, -dx))  # invert to map folder->ref

        if collected:
            dy_avg = int(round(np.mean([s[0] for s in collected])))
            dx_avg = int(round(np.mean([s[1] for s in collected])))
        else:
            dy_avg = dx_avg = 0

        shifts[cdir.name] = (dy_avg, dx_avg)

    return shifts, (H, W)

def compute_common_crop(shifts_by_c, image_shape):
    H, W = image_shape
    dy_list = [v[0] for v in shifts_by_c.values()]
    dx_list = [v[1] for v in shifts_by_c.values()]

    y0 = max([-dy for dy in dy_list])
    y1 = min([H - 1 - dy for dy in dy_list])
    x0 = max([-dx for dx in dx_list])
    x1 = min([W - 1 - dx for dx in dx_list])

    if y1 < y0 or x1 < x0:
        raise RuntimeError("No overlapping area left for cropping")

    return (int(x0), int(y0), int(x1 + 1), int(y1 + 1))

def crop_all_images(c_folders, shifts_by_c, crop_box, out_root):
    x0a, y0a, x1a, y1a = crop_box

    for cdir in c_folders:
        dy, dx = shifts_by_c[cdir.name]

        # IMPORTANT: map aligned → original coordinates by SUBTRACTING the shift
        x0 = x0a - dx
        x1 = x1a - dx
        y0 = y0a - dy
        y1 = y1a - dy

        # Keep the structure: OUT_ROOT / <SampleFolder> / <cFolder>
        out_dir = out_root / cdir.relative_to(cdir.parents[1])
        # Alternatively:
        # out_dir = out_root / cdir.parent.name / cdir.name

        out_dir.mkdir(parents=True, exist_ok=True)

        for p in sorted(cdir.iterdir()):
            if not is_tiff(p):
                continue

            with Image.open(p) as im:
                W, H = im.size

                # Clamp to image bounds for safety
                left   = max(0, min(int(x0), W))
                right  = max(left, min(int(x1), W))
                top    = max(0, min(int(y0), H))
                bottom = max(top, min(int(y1), H))

                if right - left <= 0 or bottom - top <= 0:
                    print(f"[WARN] Empty crop for {p}. Skipping.")
                    continue

                cropped = im.crop((left, top, right, bottom))
                cropped.save(out_dir / p.name, compression="tiff_deflate")

def process_root():
    groups = find_c_groups(ROOT)
    print(f"Found {len(groups)} groups.")

    for g in groups:
        c_folders = list_c_folders(g)
        shifts, shape = compute_folder_shifts(c_folders, REF_C, PATTERN)

        print(f"\nGroup: {g}")
        for k, v in shifts.items():
            print(f" {k}: dy={v[0]} dx={v[1]}")

        crop_box = compute_common_crop(shifts, shape)
        print("Crop Box:", crop_box)

        crop_all_images(c_folders, shifts, crop_box, OUT_ROOT)

    print("\nFinished!")

# RUN
process_root()

Found 2 groups.

Group: C:\Users\karako\Downloads\330\CM_TestForCrop\B02
 c1: dy=0 dx=0
 c2: dy=-77 dx=-42
 c3: dy=17 dx=21
Crop Box: (42, 77, 1031, 1037)

Group: C:\Users\karako\Downloads\330\CM_TestForCrop\C02
 c1: dy=0 dx=0
 c2: dy=-6 dx=-14
 c3: dy=0 dx=0
Crop Box: (14, 6, 1152, 1152)

Finished!
